# FIFA World Cup 2026 — Notebook 06: Model Evaluation

## About

**Purpose:** Score how good the baseline Poisson model actually is, on matches it never saw.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-07<br>
**Notes:** A model that *runs* is not a model that *works*. This notebook does a proper **time-based train/test split** — strengths are rebuilt on matches before a cutoff date, then graded on matches after it (no leakage). Predicted win/draw/loss probabilities are scored with **Brier, log loss, and Ranked Probability Score (RPS)**, checked for **calibration**, and compared against a dumb base-rate benchmark so the numbers mean something. The honest finding is expected to be that the baseline beats the dumb benchmark but is mediocre and under-confident on home wins (no home-advantage term yet) — which motivates the Elo / Dixon–Coles upgrades.<br>
**Description:** Reads `played_matches.parquet` (notebook 01); self-contained strength rebuild so it does not depend on the all-history `team_strengths.parquet`.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-06-07 | 1.0     | Ganapathy K | Initial version |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from pathlib import Path

### 1.2 Config

`CUTOFF_DATE` splits history: everything before it trains the strengths, everything after it is the held-out test. `MIN_TRAIN_GAMES` drops test matches where either side is too thin in the training data to rate reliably. The recency/shrinkage knobs match notebook 02.

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
PLAYED_MATCHES_PATH = PROCESSED_DATA_DIR / "played_matches.parquet"
EVAL_METRICS_PATH = PROCESSED_DATA_DIR / "eval_metrics.parquet"
CALIBRATION_PATH = PROCESSED_DATA_DIR / "eval_calibration.parquet"

CUTOFF_DATE = pd.Timestamp("2023-01-01")
MIN_TRAIN_GAMES = 10
HALF_LIFE_YEARS = 4
SHRINKAGE_GAMES = 30
MAX_GOALS = 10

## 2. Train/Test Split

Load all played matches and split on the cutoff date. Strengths are rebuilt **on the training half only** — the cardinal rule of evaluation is that you never grade a model on data it learned from. `build_strengths` is the notebook-02 recency-weighted + shrinkage recipe, packaged as a function so we can fit it on any slice.

In [4]:
played_matches = pd.read_parquet(PLAYED_MATCHES_PATH)

train_matches = played_matches[played_matches["date"] < CUTOFF_DATE]
test_matches = played_matches[played_matches["date"] >= CUTOFF_DATE].copy()
print(f"Train matches (before {CUTOFF_DATE.date()}): {len(train_matches)}")
print(f"Test matches  (from {CUTOFF_DATE.date()}):   {len(test_matches)}")


def build_strengths(matches, reference_date):
    home_view = matches[["date", "home_team", "home_score", "away_score"]].rename(
        columns={"home_team": "team", "home_score": "scored", "away_score": "conceded"})
    away_view = matches[["date", "away_team", "away_score", "home_score"]].rename(
        columns={"away_team": "team", "away_score": "scored", "home_score": "conceded"})
    team_matches = pd.concat([home_view, away_view], ignore_index=True)

    age_years = (reference_date - team_matches["date"]).dt.days / 365.25
    weight = 0.5 ** (age_years / HALF_LIFE_YEARS)
    team_matches["weight"] = weight
    team_matches["weighted_scored"] = weight * team_matches["scored"]
    team_matches["weighted_conceded"] = weight * team_matches["conceded"]

    weighted_baseline = team_matches["weighted_scored"].sum() / team_matches["weight"].sum()
    grouped = team_matches.groupby("team").agg(
        games=("weight", "size"),
        weight_sum=("weight", "sum"),
        scored_sum=("weighted_scored", "sum"),
        conceded_sum=("weighted_conceded", "sum"))
    grouped["attack"] = ((grouped["scored_sum"] + SHRINKAGE_GAMES * weighted_baseline)
                         / (grouped["weight_sum"] + SHRINKAGE_GAMES)) / weighted_baseline
    grouped["defence"] = ((grouped["conceded_sum"] + SHRINKAGE_GAMES * weighted_baseline)
                          / (grouped["weight_sum"] + SHRINKAGE_GAMES)) / weighted_baseline
    return grouped, weighted_baseline


reference_date = train_matches["date"].max()
strengths, baseline_goals = build_strengths(train_matches, reference_date)
print(f"Trained strengths for {len(strengths)} teams; weighted baseline = {baseline_goals:.3f}")

Train matches (before 2023-01-01): 45806
Test matches  (from 2023-01-01):   3500
Trained strengths for 333 teams; weighted baseline = 1.352


## 3. Generate Predictions

For each test match where both teams are rated (and have at least `MIN_TRAIN_GAMES` training games), run the Poisson engine to get win / draw / loss probabilities, and record the actual outcome (0 = home win, 1 = draw, 2 = away win).

In [5]:
def predict_wdl(home_team, away_team):
    home, away = strengths.loc[home_team], strengths.loc[away_team]
    expected_home = home["attack"] * away["defence"] * baseline_goals
    expected_away = away["attack"] * home["defence"] * baseline_goals
    goals = np.arange(0, MAX_GOALS + 1)
    grid = np.outer(poisson.pmf(goals, expected_home), poisson.pmf(goals, expected_away))
    p_home = np.tril(grid, -1).sum()
    p_draw = np.trace(grid)
    p_away = np.triu(grid, 1).sum()
    total = p_home + p_draw + p_away
    return p_home / total, p_draw / total, p_away / total


rated = set(strengths.index[strengths["games"] >= MIN_TRAIN_GAMES])
mask = test_matches["home_team"].isin(rated) & test_matches["away_team"].isin(rated)
eval_matches = test_matches[mask].copy()

predictions = [predict_wdl(h, a) for h, a in zip(eval_matches["home_team"], eval_matches["away_team"])]
eval_matches[["p_home", "p_draw", "p_away"]] = predictions

# actual outcome: 0 home win, 1 draw, 2 away win
outcome = np.where(eval_matches["home_score"] > eval_matches["away_score"], 0,
                   np.where(eval_matches["home_score"] == eval_matches["away_score"], 1, 2))
eval_matches["outcome"] = outcome

print(f"Scoring {len(eval_matches)} test matches "
      f"({len(test_matches) - len(eval_matches)} dropped: a side below {MIN_TRAIN_GAMES} train games)")
print("Actual outcome split:  home %.1f%% | draw %.1f%% | away %.1f%%" % tuple(
    100 * np.bincount(outcome, minlength=3) / len(outcome)))

Scoring 3475 test matches (25 dropped: a side below 10 train games)
Actual outcome split:  home 47.0% | draw 23.0% | away 30.0%


## 4. Brier Score & Log Loss

Both grade the *probabilities*, not just the top pick. **Brier** = mean squared distance between the predicted probability vector and the one-hot actual (0 = perfect, lower is better). **Log loss** punishes confident wrong calls harshly (a 1% probability on what actually happened is brutal).

In [6]:
probs = eval_matches[["p_home", "p_draw", "p_away"]].to_numpy()
actual_onehot = np.eye(3)[eval_matches["outcome"].to_numpy()]

brier = np.mean(np.sum((probs - actual_onehot) ** 2, axis=1))
picked = probs[np.arange(len(probs)), eval_matches["outcome"].to_numpy()]
log_loss = -np.mean(np.log(np.clip(picked, 1e-15, 1)))

print(f"Brier score: {brier:.4f}  (0 = perfect, lower better)")
print(f"Log loss:    {log_loss:.4f}  (lower better)")

Brier score: 0.5703  (0 = perfect, lower better)
Log loss:    0.9625  (lower better)


## 5. Ranked Probability Score (RPS)

The football-standard metric. W/D/L is **ordered** (home → draw → away), and RPS respects that: predicting a draw when it was a home win is *less wrong* than predicting an away win. It compares the cumulative probabilities, so being "close on the ladder" is rewarded. Lower is better.

In [7]:
def ranked_probability_score(probs, outcomes):
    cum_pred = np.cumsum(probs, axis=1)
    cum_actual = np.cumsum(np.eye(3)[outcomes], axis=1)
    # average of squared cumulative differences over the first r-1 = 2 categories
    return np.mean(np.sum((cum_pred[:, :-1] - cum_actual[:, :-1]) ** 2, axis=1) / 2)


rps = ranked_probability_score(probs, eval_matches["outcome"].to_numpy())
print(f"RPS: {rps:.4f}  (lower better)")

RPS: 0.1978  (lower better)


## 6. Calibration

Do the probabilities mean what they say? Bin matches by their predicted home-win probability, then check the *actual* home-win rate in each bin. A well-calibrated model sits on the diagonal — "predicted 70%" really wins ~70% of the time. Watch for the model being **under-confident on home wins**, since it has no home-advantage term.

In [8]:
bins = np.linspace(0, 1, 11)
eval_matches["home_bin"] = pd.cut(eval_matches["p_home"], bins, include_lowest=True)
calibration = eval_matches.groupby("home_bin", observed=True).agg(
    matches=("outcome", "size"),
    mean_predicted_home=("p_home", "mean"),
    actual_home_rate=("outcome", lambda s: (s == 0).mean()),
).reset_index(drop=True)
calibration["gap"] = (calibration["actual_home_rate"] - calibration["mean_predicted_home"]).round(3)
calibration.round(3)

,matches,mean_predicted_home,actual_home_rate,gap
0,25,0.065,0.040,-0.025
1,167,0.162,0.102,-0.060
2,635,0.258,0.208,-0.050
3,1100,0.352,0.405,0.053
4,940,0.445,0.580,0.134
5,436,0.540,0.768,0.229
6,132,0.640,0.909,0.269
7,28,0.735,0.929,0.193
8,11,0.844,1.000,0.156
9,1,0.907,1.000,0.093


## 7. Baseline Comparison & Verdict

A score is meaningless without a reference. The dumb benchmark: ignore the teams entirely and predict the **training-set base rates** (overall home/draw/away frequencies) for every match. If the Poisson model can't beat that, it has learned nothing. The gap between them is the model's real value.

In [9]:
train_home = home = (train_matches["home_score"] > train_matches["away_score"]).mean()
train_draw = (train_matches["home_score"] == train_matches["away_score"]).mean()
train_away = (train_matches["home_score"] < train_matches["away_score"]).mean()
base_vector = np.array([train_home, train_draw, train_away])

base_probs = np.tile(base_vector, (len(eval_matches), 1))
base_brier = np.mean(np.sum((base_probs - actual_onehot) ** 2, axis=1))
base_picked = base_probs[np.arange(len(base_probs)), eval_matches["outcome"].to_numpy()]
base_log_loss = -np.mean(np.log(np.clip(base_picked, 1e-15, 1)))
base_rps = ranked_probability_score(base_probs, eval_matches["outcome"].to_numpy())

metrics = pd.DataFrame({
    "Poisson model": [brier, log_loss, rps],
    "Base-rate benchmark": [base_brier, base_log_loss, base_rps],
}, index=["Brier", "Log loss", "RPS"])
metrics["Improvement %"] = ((metrics["Base-rate benchmark"] - metrics["Poisson model"])
                            / metrics["Base-rate benchmark"] * 100).round(1)
metrics = metrics.round(4)

metrics.to_parquet(EVAL_METRICS_PATH)
calibration.to_parquet(CALIBRATION_PATH)
print("Lower is better on all three. Positive Improvement % = model beats the dumb benchmark.\n")
print(metrics.to_string())

Lower is better on all three. Positive Improvement % = model beats the dumb benchmark.

          Poisson model  Base-rate benchmark  Improvement %
Brier            0.5703               0.6371           10.5
Log loss         0.9625               1.0553            8.8
RPS              0.1978               0.2299           14.0
